# EconomIA — Dataset de transacciones

Este notebook genera un historial financiero mensual coherente para cada usuario de `users_mejorado.csv`.

### Principios de generación

- Cada usuario tiene actividad durante los 12 meses de 2025.
- Se generan ingresos base y variables, gastos recurrentes y variables, pagos de deuda, ahorro, inversión y financiamiento del déficit.
- El gasto depende del ingreso, la situación de vida, la ocupación, la estacionalidad y el arquetipo sintético.
- El ahorro se compara con la meta declarada, pero se limita al efectivo disponible.
- La inversión solo aparece cuando existe capacidad financiera.
- El arquetipo se usa únicamente para simular comportamiento y **no se exporta en las transacciones**.
- Los IDs conservan exactamente el formato de `users_mejorado.csv`.

El archivo principal generado es `transactions_mejorado.csv`. También se exporta un archivo de auditoría mensual para comprobar la coherencia de la simulación.

In [ ]:
# =========================
# LIBRERÍAS Y CONFIGURACIÓN
# =========================

from pathlib import Path
import calendar
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

START_DATE = pd.Timestamp("2025-01-01")
END_DATE = pd.Timestamp("2025-12-31")
MONTHS = pd.date_range(START_DATE, END_DATE, freq="MS")

OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

users_df = pd.read_csv("/content/users.csv")
print(f"Usuarios cargados: {len(users_df):,}")
users_df.head(3)

Usuarios cargados: 300


,user_id,edad,sexo,ocupacion,situacion_vida,ciudad,ingreso_base,ingreso_variable,ingreso_total_estimado,meta_ahorro,frecuencia_ahorro,ratio_deuda_inicial,saldo_deuda_inicial,arquetipo_comportamiento
0,USR_00001,29,F,Médico,Vive solo,Mexicali,48369.77,2369.07,50738.84,7.56,Baja,0.5407,27434.49,Ahorrador
1,USR_00002,19,F,Estudiante,Vive con familia,Oaxaca,4530.47,228.95,4759.42,9.97,Baja,0.4357,2073.68,Equilibrado
2,USR_00003,20,M,Programador,Vive con familia,Guadalajara,50352.29,1607.84,51960.13,11.95,Alta,0.5765,29955.01,Equilibrado


In [ ]:
# ==============================================================
# CATÁLOGO COMPLETO DE CATEGORÍAS, SUBCATEGORÍAS Y DESCRIPCIONES
# ==============================================================

# N. Las categorías de consumo participan en la selección probabilística de gastos variables.

CONSUMPTION_CATEGORIES = {
    "Alimentación": ["Supermercado", "Restaurante", "Tiendas", "Comida rápida", "Alimento para mascotas"],
    "Transporte": ["Uber", "Taxi", "Combustible", "Autobús", "Tren", "Avión", "Servicio vehicular", "Verificación vehicular", "Tenencia vehicular"],
    "Salud": ["Medicinas", "Hospital", "Laboratorios", "Dentista", "Óptica", "Terapeuta", "Ejercicio", "Nutriólogo", "Veterinario"],
    "Vivienda": ["Renta", "Hipoteca", "Mantenimiento", "Predial", "Muebles", "Electrodomésticos"],
    "Educación": ["Cursos", "Universidad", "Papelería", "Material escolar"],
    "Entretenimiento": ["Suscripciones", "Cine", "Conciertos", "Museos", "Videojuegos", "Deportes", "Extras"],
    "Servicios": ["Electricidad", "Teléfono", "Gas", "Agua", "Internet", "TV de paga"],
    "Compras": ["Regalos", "Ropa", "Calzado", "Accesorios", "Electrónica", "Mascotas"],
    "Deudas":["Pago Tarjeta","Pago Préstamo","Impuestos"],
    "Finanzas":["Ahorro","Inversiones"],
    "Otros": ["Frituras", "Tintorería", "Lavandería", "Estacionamiento", "Donaciones", "Otros"]
}

# Estas categorías se generan con reglas financieras específicas.
# No deben entrar al sorteo de gastos variables.
FINANCIAL_CATEGORIES = {
    "Ingreso": ["Sueldo", "Variable"],
    "Deudas": ["Pago de tarjeta", "Pago de préstamo", "Nuevo crédito"],
    "Finanzas": ["Ahorro", "Inversiones"],
}

# Catálogo maestro que contiene todas las categorías exportables.
CATEGORIES = {**CONSUMPTION_CATEGORIES,**FINANCIAL_CATEGORIES,}

DESCRIPTIONS = {subcategory: [subcategory] for subcategories in
                CATEGORIES.values()for subcategory in subcategories}

DESCRIPTIONS.update({
    # Ingresos
    "Sueldo": ["Ingreso base", "Pago de nómina"],
    "Variable": ["Ingreso variable", "Comisión", "Honorarios", "Bono"],

    # Deudas
    "Pago de tarjeta": [
        "Pago de tarjeta", "Pago de tarjeta de crédito", "Abono a tarjeta"
    ],
    "Pago de préstamo": [
        "Pago de préstamo", "Pago de crédito", "Abono a préstamo"
    ],
    "Nuevo crédito": [
        "Disposición de crédito", "Nuevo financiamiento"
    ],

    # Finanzas
    "Ahorro": ["Ahorro", "Transferencia a ahorro", "Depósito a fondo de ahorro"],
    "Inversiones": ["Inversión", "Transferencia a inversión", "Aportación a inversión"],

    # Gastos de consumo
    "Supermercado": ["Supermercado", "Compra de alimentos", "Despensa", "Compra de víveres"],
    "Restaurante": ["Comida en restaurante", "Almuerzo fuera de casa", "Restaurante"],
    "Tiendas": ["Compra en tienda", "Tienda de abarrotes"],
    "Comida rápida": ["Comida rápida", "Comida para llevar", "Pizza", "Hamburguesa"],
    "Alimento para mascotas": ["Alimento para mascotas", "Comida para perro", "Comida para gato"],
    "Uber": ["Uber", "Viaje en Uber"],
    "Taxi": ["Taxi", "Servicio de taxi"],
    "Autobús": ["Autobús", "Transporte público"],
    "Combustible": ["Gasolina", "Combustible"],
    "Medicinas": ["Medicinas", "Medicamentos"],
    "Hospital": ["Hospital", "Consulta médica"],
    "Laboratorios": ["Laboratorio", "Estudios médicos"],
    "Terapeuta": ["Terapia", "Consulta con terapeuta"],
    "Dentista": ["Dentista", "Consulta dental"],
    "Óptica": ["Óptica", "Lentes"],
    "Ejercicio": ["Gimnasio", "Actividad física"],
    "Nutriólogo": ["Consulta con nutriólogo"],
    "Veterinario": ["Veterinario", "Consulta veterinaria"],
    "Renta": ["Pago de renta", "Renta mensual"],
    "Hipoteca": ["Pago de hipoteca"],
    "Mantenimiento": ["Mantenimiento de vivienda", "Apoyo para vivienda"],
    "Predial": ["Impuesto predial"],
    "Cursos": ["Curso", "Curso en línea"],
    "Universidad": ["Universidad", "Colegiatura"],
    "Papelería": ["Papelería"],
    "Material escolar": ["Material escolar"],
    "Suscripciones": ["Suscripción", "Servicio de streaming"],
    "Cine": ["Cine", "Entrada de cine"],
    "Conciertos": ["Concierto", "Entrada a concierto"],
    "Museos": ["Museo", "Entrada a museo"],
    "Videojuegos": ["Videojuegos"],
    "Deportes": ["Actividad deportiva", "Evento deportivo"],
    "Extras": ["Entretenimiento", "Actividad recreativa"],
    "Electricidad": ["Recibo de electricidad", "Pago de luz"],
    "Teléfono": ["Pago de teléfono"],
    "Gas": ["Pago de gas"],
    "Agua": ["Recibo de agua"],
    "Internet": ["Pago de internet"],
    "TV de paga": ["Televisión de paga"],
    "Regalos": ["Regalo", "Compra de regalo"],
    "Ropa": ["Compra de camisa","Compra de pantalón","Compra de short","Compra de sueter"],
    "Calzado": ["Compra de tenis","Compra de zapatos"],
    "Electrónica": ["Electrónica","Compra de teléfono","Compra de Laptop","Compra de audífonos","Compra de Tablet"],
    "Mascotas": ["Accesorios para mascotas"],
    "Frituras": ["Frituras", "Botana"],
    "Estacionamiento": ["Estacionamiento"],
    "Donaciones": ["Donación","Caridad","Diezmo"],
    "Otros": ["Gasto general", "Otro gasto"],
})

print("Categorías de consumo:", sorted(CONSUMPTION_CATEGORIES))
print("Categorías financieras:", sorted(FINANCIAL_CATEGORIES))


Categorías de consumo: ['Alimentación', 'Compras', 'Deudas', 'Educación', 'Entretenimiento', 'Finanzas', 'Otros', 'Salud', 'Servicios', 'Transporte', 'Vivienda']
Categorías financieras: ['Deudas', 'Finanzas', 'Ingreso']


In [ ]:
# ========================
# REGLAS DE COMPORTAMIENTO
# ========================

BASE_CATEGORY_WEIGHTS = {
    "Alimentación": 0.30,
    "Transporte": 0.15,
    "Salud": 0.08,
    "Vivienda": 0.07,
    "Educación": 0.06,
    "Entretenimiento": 0.10,
    "Servicios": 0.04,
    "Compras": 0.12,
    "Otros": 0.08,
}

ARCHETYPE_CATEGORY_MODIFIERS = {
    "Ahorrador": {"Entretenimiento": 0.65, "Compras": 0.65, "Otros": 0.75, "Alimentación": 0.95},
    "Equilibrado": {},
    "Gastador": {"Entretenimiento": 1.65, "Compras": 1.55, "Otros": 1.35, "Alimentación": 1.10},
    "Endeudado": {"Entretenimiento": 0.75, "Compras": 0.80, "Otros": 0.90},
    "Financieramente_Inestable": {"Compras": 1.45, "Entretenimiento": 1.35, "Otros": 1.50},
}

LIFE_CATEGORY_MODIFIERS = {
    "Vive con familia": {"Vivienda": 0.45, "Alimentación": 1.15},
    "Vive solo": {"Vivienda": 1.45, "Servicios": 1.25},
    "Vive con pareja": {"Vivienda": 1.15, "Alimentación": 1.10},
    "Comparte vivienda": {"Vivienda": 0.75, "Servicios": 0.85},
}

OCCUPATION_CATEGORY_MODIFIERS = {
    "Estudiante": {"Educación": 2.00, "Transporte": 1.15, "Entretenimiento": 1.15},
    "Jubilado": {"Salud": 2.00, "Transporte": 0.70, "Entretenimiento": 0.75},
    "Médico": {"Transporte": 1.10},
    "Enfermero": {"Transporte": 1.10},
    "Emprendedor": {"Transporte": 1.20, "Compras": 1.15},
    "Freelancer": {"Servicios": 1.25, "Transporte": 0.85},
}

# Porcentaje mensual de ingreso destinado a gastos de consumo.
EXPENSE_RATIO_BY_ARCHETYPE = {
    "Ahorrador": (0.52, 0.68),
    "Equilibrado": (0.62, 0.80),
    "Gastador": (0.82, 1.08),
    "Endeudado": (0.62, 0.82),
    "Financieramente_Inestable": (0.70, 1.12),
}

SAVING_RATE_BY_ARCHETYPE = {
    "Ahorrador": (0.15, 0.30),
    "Equilibrado": (0.08, 0.18),
    "Gastador": (0.01, 0.08),
    "Endeudado": (0.00, 0.07),
    "Financieramente_Inestable": (0.00, 0.20),
}

SAVING_FREQUENCY_PROB = {"Alta": 0.95, "Media": 0.72, "Baja": 0.42}

DEBT_PAYMENT_RATE = {
    "Ahorrador": (0.05, 0.10),
    "Equilibrado": (0.05, 0.09),
    "Gastador": (0.03, 0.07),
    "Endeudado": (0.08, 0.16),
    "Financieramente_Inestable": (0.03, 0.10),
}

OCCUPATION_VOLATILITY = {
    "Estudiante": 0.05,
    "Comerciante": 0.10,
    "Emprendedor": 0.16,
    "Freelancer": 0.18,
    "Chef": 0.08,
    "Diseñador": 0.08,
    "Veterinario": 0.07,
    "Abogado": 0.06,
}

PAYMENT_METHODS = ["Tarjeta de débito", "Tarjeta de crédito", "Transferencia", "Efectivo"]
CHANNELS = {
    "Tarjeta de débito": ["Terminal", "App bancaria"],
    "Tarjeta de crédito": ["Terminal", "App bancaria"],
    "Transferencia": ["App bancaria"],
    "Efectivo": ["Presencial"],
}

In [ ]:
# =====================
# FUNCIONES AUXILIARES
# =====================

def random_date_in_month(month: pd.Timestamp, preferred_day: int | None = None) -> pd.Timestamp:
    """Genera una fecha válida dentro del mes indicado."""
    days_in_month = calendar.monthrange(month.year, month.month)[1]
    if preferred_day is None:
        day = int(rng.integers(1, days_in_month + 1))
    else:
        day = min(max(1, int(round(rng.normal(preferred_day, 2)))), days_in_month)
    return pd.Timestamp(year=month.year, month=month.month, day=day)


def choose_payment() -> tuple[str, str]:
    method = random.choices(PAYMENT_METHODS, weights=[0.36, 0.27, 0.24, 0.13], k=1)[0]
    return method, random.choice(CHANNELS[method])


def seasonality_factor(category: str, month_number: int) -> float:
    """Modifica la probabilidad de ciertas categorías según el mes."""
    factor = 1.0
    if month_number == 12 and category in {"Compras", "Entretenimiento", "Alimentación"}:
        factor *= 1.35
    if month_number in {8, 9} and category == "Educación":
        factor *= 1.65
    if month_number in {4, 7} and category in {"Entretenimiento", "Transporte"}:
        factor *= 1.15
    if month_number in {1, 12} and category == "Salud":
        factor *= 1.08
    return factor


def category_weights(user: pd.Series, month_number: int) -> tuple[list[str], np.ndarray]:
    weights = BASE_CATEGORY_WEIGHTS.copy()
    modifiers = [
        ARCHETYPE_CATEGORY_MODIFIERS.get(user["arquetipo_comportamiento"], {}),
        LIFE_CATEGORY_MODIFIERS.get(user["situacion_vida"], {}),
        OCCUPATION_CATEGORY_MODIFIERS.get(user["ocupacion"], {}),
    ]
    for mapping in modifiers:
        for category, modifier in mapping.items():
            weights[category] *= modifier
    for category in weights:
        weights[category] *= seasonality_factor(category, month_number)

    categories = list(weights)
    probabilities = np.array([weights[category] for category in categories], dtype=float)
    probabilities /= probabilities.sum()
    return categories, probabilities


def add_transaction(
    transactions: list[dict],
    counter: int,
    user_id: str,
    date: pd.Timestamp,
    description: str,
    category: str,
    subcategory: str,
    amount: float,
    transaction_type: str,
    recurrent: bool,
    payment_method: str,
    channel: str,
    origin: str,
) -> int:
    """Agrega una transacción y devuelve el siguiente consecutivo."""
    if amount <= 0.005:
        return counter

    transactions.append({
        "transaction_id": f"TRX_{counter:09d}",
        "user_id": user_id,
        "fecha": pd.Timestamp(date),
        "periodo": pd.Timestamp(date).strftime("%Y-%m"),
        "descripcion": description,
        "categoria": category,
        "subcategoria": subcategory,
        "monto": round(float(amount), 2),
        "tipo": transaction_type,
        "metodo_pago": payment_method,
        "canal": channel,
        "recurrente": bool(recurrent),
        "origen_generacion": origin,
    })
    return counter + 1

In [ ]:
# ==============================
# GENERADOR DE INGRESOS Y GASTOS
# ==============================

# Genera el ingreso base y, en algunos meses, un ingreso variable.
def generate_income_transactions(user, month, transactions, counter):
    volatility = OCCUPATION_VOLATILITY.get(user["ocupacion"], 0.035)

    base_income = user["ingreso_base"] * rng.normal(1, volatility)
    base_income = max(base_income, 0)

    counter = add_transaction(
        transactions, counter, user["user_id"], random_date_in_month(month, 15),
        "Ingreso base", "Ingreso", "Sueldo", base_income,
        "Ingreso", True, "Transferencia", "App bancaria", "ingreso_base"
    )

    variable_income = 0
    receives_variable_income = user["ingreso_variable"] > 0 and rng.random() < 0.82

    if receives_variable_income:
        variable_income = user["ingreso_variable"] * rng.lognormal(-0.04, 0.30)

        counter = add_transaction(
            transactions, counter, user["user_id"], random_date_in_month(month),
            "Ingreso variable", "Ingreso", "Variable", variable_income,
            "Ingreso", False, "Transferencia", "App bancaria", "ingreso_variable"
        )

    total_income = base_income + variable_income
    return counter, total_income


# Define los pagos que normalmente se repiten cada mes.
def recurring_plan(user, monthly_income):
    plan = []
    life_situation = user["situacion_vida"]

    # Vivienda
    if life_situation == "Vive solo":
        subcategory = "Renta" if rng.random() < 0.72 else "Hipoteca"
        amount = monthly_income * rng.uniform(0.21, 0.29)
    elif life_situation == "Vive con pareja":
        subcategory = "Renta" if rng.random() < 0.58 else "Hipoteca"
        amount = monthly_income * rng.uniform(0.14, 0.21)
    elif life_situation == "Comparte vivienda":
        subcategory = "Renta"
        amount = monthly_income * rng.uniform(0.09, 0.15)
    else:
        subcategory = "Mantenimiento"
        amount = monthly_income * rng.uniform(0.035, 0.075)

    plan.append(("Vivienda", subcategory, amount, 5))

    # Servicios básicos
    services = [
        ("Electricidad", 0.012, 0.025, 8),
        ("Internet", 0.010, 0.018, 12),
        ("Teléfono", 0.008, 0.016, 18),
        ("Agua", 0.005, 0.012, 10),
        ("Gas", 0.006, 0.014, 22),
    ]

    for subcategory, min_rate, max_rate, day in services:
        if rng.random() < 0.92:
            amount = monthly_income * rng.uniform(min_rate, max_rate)
            plan.append(("Servicios", subcategory, amount, day))

    # Otros gastos recurrentes
    if rng.random() < 0.66:
        amount = monthly_income * rng.uniform(0.004, 0.012)
        plan.append(("Entretenimiento", "Suscripciones", amount, 20))

    if user["ocupacion"] == "Estudiante" and rng.random() < 0.78:
        amount = monthly_income * rng.uniform(0.07, 0.16)
        plan.append(("Educación", "Universidad", amount, 3))

    if rng.random() < 0.42:
        amount = monthly_income * rng.uniform(0.008, 0.025)
        plan.append(("Salud", "Ejercicio", amount, 7))

    return plan


# Convierte el plan recurrente en transacciones.
def generate_recurring_expenses(user, month, monthly_income, transactions, counter):
    recurring_total = 0
    plan = recurring_plan(user, monthly_income)

    for category, subcategory, base_amount, day in plan:
        amount = max(20, base_amount * rng.normal(1, 0.055))
        payment_method, channel = choose_payment()

        counter = add_transaction(
            transactions, counter, user["user_id"], random_date_in_month(month, day),
            random.choice(DESCRIPTIONS[subcategory]), category, subcategory, amount,
            "Gasto", True, payment_method, channel, "recurrente"
        )
        recurring_total += amount

    return counter, recurring_total


# Divide el presupuesto mensual entre varios gastos variables.
def generate_variable_expenses(user, month, budget, transactions, counter):
    if budget <= 10:
        return counter, 0

    transaction_count = int(8 + budget / 2500)
    transaction_count = max(6, min(transaction_count, 25))

    # Se crean proporciones aleatorias para repartir todo el presupuesto.
    proportions = rng.random(transaction_count)
    amounts = proportions / proportions.sum() * budget

    categories, probabilities = category_weights(user, month.month)
    variable_total = 0

    for amount in amounts:
        category = str(rng.choice(categories, p=probabilities))
        subcategory = random.choice(CATEGORIES[category])
        payment_method, channel = choose_payment()

        counter = add_transaction(
            transactions, counter, user["user_id"], random_date_in_month(month),
            random.choice(DESCRIPTIONS[subcategory]), category, subcategory, amount,
            "Gasto", False, payment_method, channel, "variable"
        )
        variable_total += amount

    return counter, variable_total

In [ ]:
# ==========================
# HISTORIAL MENSUAL COMPLETO
# ==========================

def generate_user_history(user, counter_start):
    transactions = []
    monthly_controls = []
    counter = counter_start
    debt_balance = float(user["saldo_deuda_inicial"])
    archetype = user["arquetipo_comportamiento"]

    for month in MONTHS:
        # 1. Ingresos
        counter, monthly_income = generate_income_transactions(
            user, month, transactions, counter
        )

        # 2. Gastos recurrentes
        counter, recurring_total = generate_recurring_expenses(
            user, month, monthly_income, transactions, counter
        )

        # 3. Gastos variables
        expense_rate = rng.uniform(*EXPENSE_RATIO_BY_ARCHETYPE[archetype])
        expected_expenses = monthly_income * expense_rate
        variable_budget = max(monthly_income * 0.08, expected_expenses - recurring_total)

        counter, variable_total = generate_variable_expenses(
            user, month, variable_budget, transactions, counter
        )
        total_expenses = recurring_total + variable_total

        # 4. Pago de deuda
        debt_payment = 0
        if debt_balance > 1:
            payment_rate = rng.uniform(*DEBT_PAYMENT_RATE[archetype])
            debt_payment = min(debt_balance, monthly_income * payment_rate)
            debt_subcategory = random.choice(["Pago de tarjeta", "Pago de préstamo"])

            counter = add_transaction(
                transactions, counter, user["user_id"], random_date_in_month(month, 25),
                random.choice(DESCRIPTIONS[debt_subcategory]),
                "Deudas", debt_subcategory, debt_payment,
                "Pago deuda", True, "Transferencia", "App bancaria", "deuda"
            )
            debt_balance -= debt_payment

        available_cash = monthly_income - total_expenses - debt_payment

        # 5. Ahorro
        saving = 0
        saving_probability = SAVING_FREQUENCY_PROB[user["frecuencia_ahorro"]]

        if available_cash > 0 and rng.random() < saving_probability:
            declared_rate = user["meta_ahorro"] / 100
            behavior_rate = rng.uniform(*SAVING_RATE_BY_ARCHETYPE[archetype])
            saving_rate = (declared_rate + behavior_rate) / 2
            saving_rate = max(0, min(saving_rate, 0.35))

            saving = min(available_cash * 0.82, monthly_income * saving_rate)

            if saving > 20:
                counter = add_transaction(
                    transactions, counter, user["user_id"], random_date_in_month(month, 27),
                    random.choice(DESCRIPTIONS["Ahorro"]),
                    "Finanzas", "Ahorro", saving,
                    "Ahorro", True, "Transferencia", "App bancaria", "ahorro"
                )
                available_cash -= saving

        # 6. Inversión
        investment = 0
        investment_probability = {
            "Ahorrador": 0.42,
            "Equilibrado": 0.24,
            "Gastador": 0.08,
            "Endeudado": 0.04,
            "Financieramente_Inestable": 0.10,
        }[archetype]

        can_invest = available_cash > monthly_income * 0.04
        if can_invest and rng.random() < investment_probability:
            investment = min(available_cash * 0.55, monthly_income * rng.uniform(0.02, 0.07))

            if investment > 20:
                counter = add_transaction(
                    transactions, counter, user["user_id"], random_date_in_month(month, 28),
                    random.choice(DESCRIPTIONS["Inversiones"]),
                    "Finanzas", "Inversiones", investment,
                    "Inversión", False, "Transferencia", "App bancaria", "inversion"
                )
                available_cash -= investment

        # 7. Financiamiento cuando los gastos superan el ingreso disponible
        financing = 0
        if available_cash < 0:
            financing = abs(available_cash)

            counter = add_transaction(
                transactions, counter, user["user_id"], random_date_in_month(month, 29),
                random.choice(DESCRIPTIONS["Nuevo crédito"]),
                "Deudas", "Nuevo crédito", financing,
                "Financiamiento", False, "Transferencia", "App bancaria", "financiamiento"
            )
            debt_balance += financing
            available_cash = 0

        # Registro de control para revisar el balance de cada mes
        monthly_controls.append({
            "user_id": user["user_id"],
            "periodo": month.strftime("%Y-%m"),
            "ingresos": monthly_income,
            "gastos": total_expenses,
            "pagos_deuda": debt_payment,
            "ahorro": saving,
            "inversion": investment,
            "financiamiento": financing,
            "saldo_caja": available_cash,
            "saldo_deuda_final": debt_balance,
        })

    return transactions, monthly_controls, counter

In [ ]:
# =================
# GENERACIÓN MASIVA
# =================

all_transactions = []
all_controls = []
transaction_counter = 1

for _, user in users_df.iterrows():
    user_transactions, user_controls, transaction_counter = generate_user_history(
        user, transaction_counter
    )
    all_transactions.extend(user_transactions)
    all_controls.extend(user_controls)

transactions_df = pd.DataFrame(all_transactions)
generation_control_df = pd.DataFrame(all_controls)

print(f"Transacciones generadas: {len(transactions_df):,}")
print(f"Registros de control mensual: {len(generation_control_df):,}")

Transacciones generadas: 90,684
Registros de control mensual: 3,600


In [ ]:
# ==================================
# VARIABLES TEMPORALES Y ORDEN FINAL
# ==================================

# Convertir la fecha al formato correcto.
transactions_df["fecha"] = pd.to_datetime(transactions_df["fecha"])

# Crear columnas útiles para el análisis mensual.
transactions_df["anio"] = transactions_df["fecha"].dt.year
transactions_df["mes"] = transactions_df["fecha"].dt.month

month_names = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
transactions_df["nombre_mes"] = transactions_df["mes"].map(month_names)

# Orden final de las columnas del CSV.
final_columns = [
    "transaction_id", "user_id", "fecha", "periodo", "descripcion",
    "categoria", "subcategoria", "monto", "tipo", "metodo_pago",
    "canal", "recurrente", "origen_generacion", "mes", "anio", "nombre_mes",
]

transactions_df = transactions_df[final_columns]
transactions_df = transactions_df.sort_values(
    by=["user_id", "fecha", "transaction_id"]
).reset_index(drop=True)

transactions_df.sample(10)

,transaction_id,user_id,fecha,periodo,descripcion,categoria,subcategoria,monto,tipo,metodo_pago,canal,recurrente,origen_generacion,mes,anio,nombre_mes
33250,TRX_000033261,USR_00110,2025-09-02,2025-09,Material escolar,Educación,Material escolar,1928.47,Gasto,Transferencia,App bancaria,False,variable,9,2025,Septiembre
43893,TRX_000043894,USR_00146,2025-03-06,2025-03,Recibo de electricidad,Servicios,Electricidad,672.27,Gasto,Tarjeta de crédito,App bancaria,True,recurrente,3,2025,Marzo
30511,TRX_000030518,USR_00101,2025-11-23,2025-11,Abono a préstamo,Deudas,Pago de préstamo,2835.86,Pago deuda,Transferencia,App bancaria,True,deuda,11,2025,Noviembre
11038,TRX_000011037,USR_00038,2025-10-10,2025-10,Recibo de agua,Servicios,Agua,69.79,Gasto,Transferencia,App bancaria,True,recurrente,10,2025,Octubre
82336,TRX_000082340,USR_00271,2025-11-09,2025-11,Otro gasto,Otros,Otros,568.49,Gasto,Tarjeta de débito,Terminal,False,variable,11,2025,Noviembre
6399,TRX_000006402,USR_00022,2025-07-25,2025-07,Nuevo financiamiento,Deudas,Nuevo crédito,1918.73,Financiamiento,Transferencia,App bancaria,False,financiamiento,7,2025,Julio
62269,TRX_000062269,USR_00206,2025-06-27,2025-06,Actividad recreativa,Entretenimiento,Extras,1843.34,Gasto,Transferencia,App bancaria,False,variable,6,2025,Junio
25891,TRX_000025888,USR_00088,2025-04-11,2025-04,Pago de luz,Servicios,Electricidad,427.04,Gasto,Transferencia,App bancaria,True,recurrente,4,2025,Abril
1994,TRX_000001988,USR_00007,2025-02-17,2025-02,Pago de teléfono,Servicios,Teléfono,385.33,Gasto,Tarjeta de débito,Terminal,True,recurrente,2,2025,Febrero
52642,TRX_000052646,USR_00175,2025-07-19,2025-07,Comida rápida,Alimentación,Comida rápida,589.78,Gasto,Tarjeta de crédito,App bancaria,False,variable,7,2025,Julio


In [ ]:
# ============
# VALIDACIONES
# ============

# Esta función muestra un mensaje más claro cuando una validación falla.
def validate(condition, message):
    if not condition:
        raise ValueError(message)

# Generales
validate(not transactions_df.empty, "No se generaron transacciones.")
validate(transactions_df["transaction_id"].is_unique, "Hay IDs de transacción repetidos.")
validate(transactions_df["monto"].gt(0).all(), "Todos los montos deben ser positivos.")
validate(transactions_df.isna().sum().sum() == 0, "Existen valores nulos.")

# Categorías permitidas
validate(set(transactions_df["categoria"]).issubset(CATEGORIES), "Existe una categoría no registrada.")

# Cada usuario debe tener información de los 12 meses.
months_per_user = transactions_df.groupby("user_id")["periodo"].nunique()
validate(months_per_user.eq(12).all(), "Hay usuarios sin transacciones en los 12 meses.")

print("Validaciones superadas correctamente.")
print(f"Usuarios únicos: {transactions_df['user_id'].nunique():,}")
print(f"Transacciones únicas: {transactions_df['transaction_id'].nunique():,}")
print()
print("Tipos de movimiento:")
print(transactions_df["tipo"].value_counts())

Validaciones superadas correctamente.
Usuarios únicos: 300
Transacciones únicas: 90,684

Tipos de movimiento:
tipo
Gasto             78848
Ingreso            6535
Ahorro             2162
Pago deuda         1985
Inversión           635
Financiamiento      519
Name: count, dtype: int64


In [ ]:
# =================================
# VALIDACIÓN DE BALANCE USUARIO-MES
# =================================

# Sumar los movimientos de cada usuario en cada mes.
monthly_validation = transactions_df.pivot_table(
    index=["user_id", "periodo"],
    columns="tipo",
    values="monto",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Asegurar que todas las columnas existan, aunque algún tipo no aparezca.
movement_columns = [
    "Ingreso", "Financiamiento", "Gasto",
    "Pago deuda", "Ahorro", "Inversión",
]

for column in movement_columns:
    if column not in monthly_validation.columns:
        monthly_validation[column] = 0

# Entradas menos salidas.
monthly_validation["saldo_calculado"] = (
    monthly_validation["Ingreso"]
    + monthly_validation["Financiamiento"]
    - monthly_validation["Gasto"]
    - monthly_validation["Pago deuda"]
    - monthly_validation["Ahorro"]
    - monthly_validation["Inversión"]
)

expected_rows = len(users_df) * len(MONTHS)
validate(len(monthly_validation) == expected_rows, "Faltan registros usuario-mes.")
validate(monthly_validation["Ingreso"].gt(0).all(), "Hay meses sin ingresos.")

print("Balance usuario-mes validado correctamente.")
print(f"Meses con financiamiento: {(monthly_validation['Financiamiento'] > 0).sum():,}")
print(f"Meses con ahorro: {(monthly_validation['Ahorro'] > 0).sum():,}")
print(f"Meses con inversión: {(monthly_validation['Inversión'] > 0).sum():,}")

monthly_validation.head()

Balance usuario-mes validado correctamente.
Meses con financiamiento: 519
Meses con ahorro: 2,162
Meses con inversión: 635


tipo,user_id,periodo,Ahorro,Financiamiento,Gasto,Ingreso,Inversión,Pago deuda,saldo_calculado
0,USR_00001,2025-01,7012.14,0.0,30763.14,48885.64,1120.22,2519.61,7470.53
1,USR_00001,2025-02,0.00,0.0,29015.43,50997.94,3126.92,3395.36,15460.23
2,USR_00001,2025-03,9120.49,0.0,32018.07,50068.40,0.00,4162.85,4766.99
3,USR_00001,2025-04,0.00,0.0,31435.90,51106.18,0.00,2808.57,16861.71
4,USR_00001,2025-05,6957.61,0.0,32659.92,49656.67,1360.05,4713.33,3965.76


In [ ]:
# ================================
# VALIDACIÓN DE CATEGORÍAS Y TIPOS
# ================================

category_validation = (
    transactions_df
    .groupby(["categoria", "tipo"], observed=True)
    .agg(
        transacciones=("transaction_id", "count"),
        monto_total=("monto", "sum"),
        usuarios=("user_id", "nunique"),
    ).reset_index().sort_values(["categoria", "tipo"])
)

category_validation["monto_total"] = category_validation["monto_total"].round(2)

print("Resumen particular de Deudas y Finanzas:")
display(category_validation[category_validation["categoria"].isin(["Deudas", "Finanzas"])].reset_index(drop=True))

category_validation


Resumen particular de Deudas y Finanzas:


,categoria,tipo,transacciones,monto_total,usuarios
0,Deudas,Financiamiento,519,1332514.57,97
1,Deudas,Pago deuda,1985,5016958.92,300
2,Finanzas,Ahorro,2162,11878787.73,298
3,Finanzas,Inversión,635,1143938.37,193


,categoria,tipo,transacciones,monto_total,usuarios
0,Alimentación,Gasto,17262,2.381339e+07,300
1,Compras,Gasto,6479,9.386348e+06,300
2,Deudas,Financiamiento,519,1.332515e+06,97
3,Deudas,Pago deuda,1985,5.016959e+06,300
4,Educación,Gasto,3556,4.732155e+06,300
5,Entretenimiento,Gasto,7789,8.553875e+06,300
6,Finanzas,Ahorro,2162,1.187879e+07,298
7,Finanzas,Inversión,635,1.143938e+06,193
8,Ingreso,Ingreso,6535,1.430674e+08,300
9,Otros,Gasto,4260,6.025384e+06,300


In [ ]:
# =================
# EXPORTAR DATASETS
# =================

transactions_path = OUTPUT_DIR / "transactions.csv"
control_path = OUTPUT_DIR / "transaction_generation_control.csv"
category_audit_path = OUTPUT_DIR / "transaction_category_validation.csv"

transactions_df.to_csv(transactions_path, index=False, encoding="utf-8-sig")
generation_control_df.to_csv(control_path, index=False, encoding="utf-8-sig")
category_validation.to_csv(category_audit_path, index=False, encoding="utf-8-sig")

print(f"Transacciones generadas en: {transactions_path}")
print(f"Control mensual generado en: {control_path}")
print(f"Auditoría generada en: {category_audit_path}")
print(f"Dimensiones finales: {transactions_df.shape}")

Transacciones generadas en: /content/transactions.csv
Control mensual generado en: /content/transaction_generation_control.csv
Auditoría generada en: /content/transaction_category_validation.csv
Dimensiones finales: (90684, 16)
